# Day 10-11 — 模型评估 & 特征重要性分析

**内容**:
- ROC曲线 & Precision-Recall曲线对比
- 混淆矩阵
- 特征重要性 (内置 + Permutation Importance)
- 按股票/按年份的子集分析
- 稳健性检验 (不同crash阈值: 10% vs 15% vs 20%)

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

OUT_DIR = r"C:\Users\1\Desktop\项目\stock-data"
MODEL_DIR = r"C:\Users\1\Desktop\项目\models"
FIG_DIR = r"C:\Users\1\Desktop\项目\figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ==========================================
# 第1步: 加载数据和模型
# ==========================================

df = pd.read_csv(os.path.join(OUT_DIR, "day4_features.csv"))
df["Date"] = pd.to_datetime(df["Date"])

LABEL_COL = "crash_binary"  # MDD <= -15% = crash

with open(os.path.join(OUT_DIR, "v2_feature_list.txt"), "r", encoding="utf-8") as f:
    feature_cols = [l.strip() for l in f if l.strip()]
feature_cols = [c for c in feature_cols if c in df.columns]

# 时序划分 (与 day8 一致)
val_end = pd.Timestamp("2023-12-31")
test_idx = df[df["Date"] > val_end].index

X_test = df.loc[test_idx, feature_cols].values.astype(np.float32)
y_test = df.loc[test_idx, LABEL_COL].values.astype(int)

print(f"测试集: {X_test.shape}")
print(f"Crash比例: {y_test.mean():.2%}")

# 加载所有模型 (先尝试 day8 输出, 再尝试 main_pipeline.py 输出)
models = {}
MODEL_NAMES = {
    "RandomForest": ["RandomForest_final.pkl", "RF_v2_final.pkl"],
    "GBDT":         ["GBDT_final.pkl",         "GBDT_v2_final.pkl"],
    "XGBoost":      ["XGBoost_final.pkl",      "XGB_v2_final.pkl"],
    "LightGBM":     ["LightGBM_final.pkl",     "LGBM_v2_final.pkl"],
}

for name, paths in MODEL_NAMES.items():
    for p in paths:
        full_path = os.path.join(MODEL_DIR, p)
        if os.path.exists(full_path):
            with open(full_path, "rb") as f:
                models[name] = pickle.load(f)
            print(f"加载 {name} ({p}) 成功")
            break

print(f"\n共 {len(models)} 个模型")

# 检测是否为多分类模型 (predict_proba 输出 > 2 列)
if models:
    first_model = list(models.values())[0]
    prob_shape = first_model.predict_proba(X_test[:1]).shape[1]
    IS_MULTICLASS = (prob_shape > 2)
    print(f"模型类型: {'多分类(4类)' if IS_MULTICLASS else '二分类'} (输出{prob_shape}列)")
else:
    IS_MULTICLASS = False

def get_crash_prob(model, X):
    """获取崩盘概率: 二分类用[:,1], 多分类用[:,2]+[:,3]"""
    prob = model.predict_proba(X)
    if prob.shape[1] > 2:
        return prob[:, 2] + prob[:, 3]  # Crash + Severe
    return prob[:, 1]

In [ ]:
# ==========================================
# 第2步: ROC曲线对比
# ==========================================

fig, ax = plt.subplots(figsize=(10, 8))

colors = {"RandomForest": "green", "GBDT": "orange",
          "XGBoost": "blue", "LightGBM": "red"}

for name, model in models.items():
    y_prob = get_crash_prob(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2, color=colors.get(name, "gray"),
            label=f"{name} (AUC={auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — 股价崩盘预测 (Test Set 2024-2025)", fontsize=14)
ax.legend(fontsize=11, loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "roc_curves.png"), dpi=150)
plt.show()

In [ ]:
# ==========================================
# 第3步: Precision-Recall 曲线 (更适合不平衡数据)
# ==========================================

fig, ax = plt.subplots(figsize=(10, 8))

for name, model in models.items():
    y_prob = get_crash_prob(model, X_test)
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(recall, precision, lw=2, color=colors.get(name, "gray"),
            label=f"{name} (AP={ap:.4f})")

no_skill = y_test.mean()
ax.axhline(y=no_skill, color="k", ls="--", alpha=0.3,
           label=f"Baseline (crash%={no_skill:.2%})")
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves (Test Set)", fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "pr_curves.png"), dpi=150)
plt.show()

In [ ]:
# ==========================================
# 第4步: 混淆矩阵 (最佳模型)
# ==========================================

# 找出AUC最高的模型
best_name = None
best_auc = 0
best_prob = None
for name, model in models.items():
    y_prob = get_crash_prob(model, X_test)
    auc = roc_auc_score(y_test, y_prob)
    if auc > best_auc:
        best_auc = auc
        best_name = name
        best_prob = y_prob

y_pred = (best_prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["No Crash", "Crash"],
            yticklabels=["No Crash", "Crash"])
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title(f"Confusion Matrix — {best_name} (AUC={best_auc:.4f})", fontsize=13)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Crash", "Crash"]))

In [ ]:
# ==========================================
# 第5步: 特征重要性 (内置 + Permutation)
# ==========================================

fig, axes = plt.subplots(2, 2, figsize=(16, 18))

for idx, (name, model) in enumerate(models.items()):
    ax = axes[idx // 2, idx % 2]
    
    # 内置特征重要性
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        continue
    
    # Top 15
    indices = np.argsort(importances)[-15:]
    top_features = [feature_cols[i] for i in indices][::-1]
    top_importances = importances[indices][::-1]
    
    colors_bar = ["coral" if "mdd" in f.lower() or "crash" in f.lower()
                  else "steelblue" for f in top_features]
    ax.barh(range(len(top_features)), top_importances, color=colors_bar)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features, fontsize=9)
    ax.set_xlabel("Importance", fontsize=11)
    ax.set_title(f"{name} — Top 15 Feature Importance", fontsize=12)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "feature_importance.png"), dpi=150)
plt.show()

In [ ]:
# ==========================================
# 第6步: Permutation Importance (模型无关)
# ==========================================

if best_name:
    best_model = models[best_name]
    
    print(f"计算 {best_name} Permutation Importance (可能需要几分钟)...")
    perm_result = permutation_importance(
        best_model, X_test, y_test,
        scoring="roc_auc", n_repeats=5,
        random_state=42, n_jobs=-1
    )
    
    perm_imp = perm_result.importances_mean
    perm_std = perm_result.importances_std
    
    # Top 20
    top_idx = np.argsort(perm_imp)[-20:][::-1]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(
        range(len(top_idx)),
        perm_imp[top_idx],
        xerr=perm_std[top_idx],
        color="steelblue", alpha=0.8, capsize=3
    )
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels([feature_cols[i] for i in top_idx], fontsize=10)
    ax.set_xlabel("AUC Decrease", fontsize=12)
    ax.set_title(f"Permutation Importance — {best_name} (Test Set)", fontsize=14)
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "permutation_importance.png"), dpi=150)
    plt.show()

In [ ]:
# ==========================================
# 第7步: 按股票分组评估
# ==========================================

df_test = df.iloc[test_idx].copy()

for name, model in models.items():
    df_test[f"prob_{name}"] = get_crash_prob(model, X_test)

# 每只股票的AUC
print("=" * 60)
print("按股票 AUC 对比")
print("=" * 60)

symbol_auc = []
for sym in df_test["symbol"].unique():
    mask = df_test["symbol"] == sym
    y_sub = df_test.loc[mask, LABEL_COL].values
    row = {"symbol": sym, "samples": mask.sum(), "crash%": y_sub.mean()}
    for name in models:
        prob_col = f"prob_{name}"
        if prob_col in df_test.columns:
            row[f"AUC_{name}"] = roc_auc_score(y_sub, df_test.loc[mask, prob_col])
    symbol_auc.append(row)

df_symbol_auc = pd.DataFrame(symbol_auc).set_index("symbol")
print(df_symbol_auc.round(3).to_string())

df_symbol_auc.to_csv(os.path.join(OUT_DIR, "day10_auc_by_symbol.csv"))
print("\n已保存.")

In [ ]:
# ==========================================
# 第8步: 稳健性检验 — 不同阈值
# ==========================================

print("=" * 60)
print("稳健性检验: 不同Crash阈值下的AUC")
print("=" * 60)

# 从 future_mdd_20 动态构造不同阈值的标签
robust_results = []

for th in [0.10, 0.15, 0.20]:
    col_name = f"crash_{int(th*100)}"
    df[col_name] = (df["future_mdd_20"] <= -th).astype(int)
    
    y_thresh = df.loc[test_idx, col_name].values.astype(int)
    
    if y_thresh.sum() == 0:
        print(f"  {col_name}: 无正样本, 跳过")
        continue
    
    for name, model in models.items():
        y_prob = get_crash_prob(model, X_test)
        auc = roc_auc_score(y_thresh, y_prob)
        robust_results.append({
            "threshold": col_name,
            "model": name,
            "AUC": auc,
            "crash_count": y_thresh.sum(),
            "crash_pct": y_thresh.mean(),
        })

df_robust = pd.DataFrame(robust_results)
print(df_robust.pivot(index="model", columns="threshold", values="AUC").round(4).to_string())

df_robust.to_csv(os.path.join(OUT_DIR, "day10_robustness.csv"), index=False)
print("\nDay10-11 完成!")